In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
    matthews_corrcoef
)



In [ ]:
# =========================
# 1. Load Data
# =========================

def load_data(path):
    df = pd.read_csv(path)
    return df



In [ ]:
# =========================
# 2. Basic Cleaning
# =========================

def clean_data(df):
    # Drop constant columns
    nunique = df.nunique()
    df = df.loc[:, nunique > 1]

    # Drop non-numeric except 'sample' and 'binding'
    keep_cols = ['sample', 'binding']
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df = df[keep_cols + [c for c in numeric_cols if c not in keep_cols]]

    # Convert binding to int
    df['binding'] = df['binding'].astype(int)

    return df



In [ ]:
# =========================
# 3. Basic Stats
# =========================

def basic_stats(df):
    print("\nClass distribution:")
    print(df['binding'].value_counts(normalize=True))

    print("\nMissing values (%):")
    print((df.isna().mean() * 100).sort_values(ascending=False).head(20))



In [ ]:
# =========================
# 4. Distribution Plots
# =========================

def plot_distributions(df, output_dir="plots"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    metrics = [c for c in df.columns if c not in ['sample', 'binding']]

    for col in metrics:
        plt.figure()
        sns.violinplot(data=df, x='binding', y=col)
        plt.title(f"{col} distribution")
        plt.savefig(f"{output_dir}/{col}_violin.png")
        plt.close()



In [ ]:
# =========================
# 5. Metric Evaluation
# =========================

def evaluate_metric(y_true, scores):
    # Handle NaNs
    mask = ~np.isnan(scores)
    y_true = y_true[mask]
    scores = scores[mask]

    if len(np.unique(y_true)) < 2:
        return None

    roc = roc_auc_score(y_true, scores)
    pr = average_precision_score(y_true, scores)

    # Find best F1 threshold
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    # MCC at best threshold
    preds = (scores >= best_thresh).astype(int)
    mcc = matthews_corrcoef(y_true, preds)

    return {
        'roc_auc': roc,
        'pr_auc': pr,
        'best_f1': best_f1,
        'best_threshold': best_thresh,
        'mcc': mcc
    }



In [ ]:
# =========================
# 6. Evaluate All Metrics
# =========================

def evaluate_all(df):
    results = {}
    y = df['binding'].values

    for col in df.columns:
        if col in ['sample', 'binding']:
            continue

        scores = df[col].values
        res = evaluate_metric(y, scores)

        if res is not None:
            results[col] = res

    results_df = pd.DataFrame(results).T
    results_df = results_df.sort_values('pr_auc', ascending=False)

    return results_df



In [ ]:
# =========================
# 7. Top-K Enrichment
# =========================

def top_k_enrichment(df, k=10):
    y = df['binding']
    enrichment = {}

    for col in df.columns:
        if col in ['sample', 'binding']:
            continue

        tmp = df[[col, 'binding']].dropna()
        tmp = tmp.sort_values(col, ascending=False).head(k)
        enrichment[col] = tmp['binding'].mean()

    return pd.Series(enrichment).sort_values(ascending=False)




In [ ]:
# =========================
# 8. Correlation Analysis
# =========================

def plot_correlation(df, output_path="correlation.png"):
    corr = df.drop(columns=['sample', 'binding']).corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title("Feature Correlation")
    plt.savefig(output_path)
    plt.close()



In [ ]:
# =========================
# 9. Main Pipeline
# =========================

def main(path):
    df = load_data(path)
    df = clean_data(df)

    basic_stats(df)

    print("\nRunning evaluation...")
    results = evaluate_all(df)
    print(results.head(20))

    print("\nTop-K enrichment:")
    enrichment = top_k_enrichment(df, k=10)
    print(enrichment.head(20))

    plot_distributions(df)
    plot_correlation(df)

    results.to_csv("metric_evaluation.csv")
    enrichment.to_csv("topk_enrichment.csv")

    print("\nDone.")


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=str, required=True, help="Path to prediction.csv")
    args = parser.parse_args()

    main(args.input)